# Human-in-the-Loop: Pre-Action Gates, Risk Tiering, and Audit Logging

The README for this lesson introduces Human-in-the-Loop with a short snippet that asks the user `APPROVE` or `REJECT` after the agent has already produced a response. That pattern is a fine starting point, but production HITL implementations commonly need three additional pieces:

1. A **pre-action gate** that runs **before** the agent executes a risky step, so cost, irreversibility, and latency stay under control.
2. **Risk tiering**, so low-risk actions auto-execute, medium-risk actions are batch-approved, and only high-risk actions block on a human.
3. An **audit log plus revision loop**, so every gate decision is recorded as JSONL, and a rejection re-prompts the agent with a structured reason instead of just printing `Revising...`.

This notebook builds each of these on top of the same primitives as `06-system-message-framework.ipynb`. It runs end-to-end in `DEMO_MODE = True` (no interactive input needed) or with real `input()` prompts when `DEMO_MODE = False`. Note: in DEMO_MODE the third goal's retry is scripted so the loop mechanics are visible end-to-end. Real revision-driven re-classification requires `DEMO_MODE = False` and an operator.

**Out of scope (handled in other lessons):** authentication and access control (Lesson 06 README threat 2), tool-call middleware (Lesson 14 MAF deep dive), multi-agent debate patterns.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

GATE_LOG_PATH = Path("gate_log.jsonl")

token = os.environ.get("GITHUB_TOKEN", "")
endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Pattern 1: Pre-action gate

The README's HITL snippet calls the agent first, then asks the user to approve the output. That is a **post-action** flow. The agent has already executed, so the LLM call cost is already paid, and any side effect (sent email, written database row, posted comment) has already happened.

A **pre-action** flow inserts the gate before the agent runs the risky step. The agent proposes the action, the gate decides whether to execute, and only on approval does the side effect occur.

| Aspect | Post-action approval (README snippet) | Pre-action gate (this notebook) |
|---|---|---|
| When does approval run? | After the agent has produced output | Before any side-effect executes |
| LLM cost on rejection | Already paid | Paid only for the proposal, not the action |
| Irreversible side effects | Possible (the action already happened) | Prevented |
| Audit clarity | Approval is a print statement | Approval is a JSON record with timestamp, action, reason |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        decision = raw if raw in {"approve", "deny", "escalate"} else "deny"
        reason = "operator input" if raw else "no input received, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Pattern 2: Risk tiering

Not every action needs human approval. A read-only lookup against a public API has different stakes than sending a customer email. Treating both the same wastes operator attention and slows the agent.

A simple 3-tier model:

| Tier | Examples | Approval flow |
|---|---|---|
| `low` (read-only) | Search a knowledge base, look up flight options, fetch a public web page | Auto-execute, logged for audit |
| `medium` (cheap mutation) | Cache a result, increment a counter, schedule a reminder | Auto-execute, but batched daily review |
| `high` (external-facing or irreversible) | Send an email, charge a card, post to a public channel | Block on human approval |

This is one tiering. Production systems often use more granular tiers (e.g., AWS IAM permission levels, role-based access tiers). The 3-tier version below is the smallest useful version for an agent that mixes read-only and side-effecting actions.

The classifier below uses keyword heuristics so the demo stays deterministic and cheap. In a production system you would swap this for a learned classifier or a policy engine.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier == "low":
        return {
            "decision": "approve",
            "reason": "auto-approved (low risk)",
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    if tier == "medium":
        return {
            "decision": "approve",
            "reason": "auto-approved (medium risk, queued for batched review)",
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Pattern 3: Audit log and revision loop

A `print("Response approved.")` is not an audit log. For trust, every gate decision should be recorded as a structured event that you can later query, replay, or attach to an incident review.

Two pieces:

1. **Append-only JSONL.** One line per decision, with timestamp, action, tier, decision, reason. Easy to grep, easy to ship to a real log store later.
2. **Revision loop on rejection.** When the gate returns `deny`, the agent re-prompts itself with the rejection reason in context, so the next proposal can avoid the problem.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.

if GATE_LOG_PATH.exists():
    GATE_LOG_PATH.unlink()  # fresh log for the demo

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print("\n=== Audit log ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Additional resources

Several other public projects implement variations of these HITL patterns. Compare approaches to find what fits your stack:

- **LangChain** human-in-the-loop tool wrappers ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): drop-in tool wrappers that pause execution for human input.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ restructured this): uses an agent role specifically to represent the human in multi-agent conversations.
- **Semantic Kernel** function filters ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware that runs around every function call, suitable for gating logic.

Each project handles the three sub-patterns differently: LangChain wraps them as tools, AutoGen uses an agent role, Semantic Kernel uses middleware filters. Read one or two implementations end-to-end before picking a design for your own agent.
